# Nifty 50 Swing Pipeline — Colab GPU Training

Trains the XGBoost swing-prediction pipeline on Colab's GPU instead of your laptop.

**Before running:** set the runtime to GPU — `Runtime ▸ Change runtime type ▸ Hardware accelerator: GPU`.

Then run the cells top to bottom.

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi -L || echo 'NO GPU — set Runtime > Change runtime type > GPU, then rerun'

## 2. Clone the repo

If the repo is **private**, replace the URL with `https://<TOKEN>@github.com/...` using a GitHub personal access token.

In [ ]:
import os

REPO_URL = 'https://github.com/aditya33agrawal/XGBoost-Swing-Trading-Prediction-System.git'
BRANCH   = 'main'  # change if you push to a different branch
REPO_DIR = '/content/swing'

if not os.path.exists(REPO_DIR):
    !git clone --branch $BRANCH $REPO_URL $REPO_DIR
else:
    !cd $REPO_DIR && git pull

# The pipeline package lives at the repo root (run_pipeline.py + src/).
%cd $REPO_DIR

## 3. Install dependencies

Colab already ships a CUDA-enabled `xgboost`, so this is quick. We skip the heavy/optional `mlflow` + `duckdb` (the pipeline degrades gracefully without them).

In [ ]:
!pip install -q 'xgboost>=2.0' optuna pandas-ta yfinance shap scipy scikit-learn pyarrow
import xgboost as xgb
print('xgboost', xgb.__version__)

## 4. Train on the GPU

`--device auto` detects the GPU automatically; the logs will say `device=cuda`.
Set `LOG_LEVEL=DEBUG` for even more detail. Lower `--trials` for a quick test run.

In [ ]:
import os
os.environ['LOG_LEVEL'] = 'INFO'   # set to 'DEBUG' for more

!python run_pipeline.py --trials 50 --device auto --log-level INFO

## 5. (Optional) Save the trained artifacts to Google Drive

Colab wipes `/content` when the runtime disconnects. Copy `models/` and `data/` to Drive to keep them.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/swing_outputs
!cp -r /content/swing/models /content/drive/MyDrive/swing_outputs/ 2>/dev/null || true
!cp -r /content/swing/data   /content/drive/MyDrive/swing_outputs/ 2>/dev/null || true
print('Saved to Drive ▸ swing_outputs')